# Day 2 | Notebook 6: FastAPI RAG Client
**Focus:** `httpx` async client, load testing, cache warmup, performance analysis

---
## 📌 Learning Objectives
1. Call a real production RAG API from a Jupyter notebook
2. Understand the full MISS → PDR → LLM → Cache → HIT cycle
3. Measure real-world latency: p50, p95, p99
4. Find the API's performance limits using a load test
---

In [3]:
# ─── INSTRUCTOR SETTINGS ─────────────────────────────────────────
SHOW_INSIGHTS = False
API_URL = "http://localhost:8000"
# ─────────────────────────────────────────────────────────────────
import httpx
import asyncio
import time
import statistics

# Quick connectivity check
try:
    r = httpx.get(f"{API_URL}/health", timeout=5)
    r.raise_for_status()
    print("✅ FastAPI service is reachable")
    print(f"   Status: {r.json()}")
except Exception as e:
    print(f"❌ Cannot reach API: {e}")
    print(f"   Ensure docker-compose.day2.yml is running")
    print(f"   Run: docker compose -f day2/docker-compose.day2.yml up -d")

✅ FastAPI service is reachable
   Status: {'status': 'online', 'redis': 'connected', 'model': 'sentence-transformers/all-MiniLM-L6-v2', 'version': '2.0.0'}


## 📌 Concept: Microservice Architecture

```
┌─────────────────────────────────────────────────────────┐
│              Docker Network (day2_default)               │
│                                                          │
│  ┌──────────────────┐      ┌───────────────────────┐    │
│  │  fastapi-service │ ──── │    redis-stack         │    │
│  │  Port 8000       │      │    Port 6379           │    │
│  │  (app.py)        │      │    (PDR + Cache + Mem) │    │
│  └──────────────────┘      └───────────────────────┘    │
│           ▲                                              │
│           │ HTTP                                         │
│  ┌────────────────────┐                                 │
│  │  jupyter-sandbox   │                                  │
│  │  Port 8888         │                                  │
│  │  (this notebook)   │                                  │
│  └────────────────────┘                                  │
└─────────────────────────────────────────────────────────┘
```

**Why separate services?**
- The API can be **scaled horizontally** (add more containers) independently of notebooks
- The API can be called from **any client** (browser, mobile app, other microservices)
- **Fault isolation**: if the notebook crashes, the API keeps running


In [3]:
# Cell 1: GET /health — check all systems
resp = httpx.get(f"{API_URL}/health")
health = resp.json()

print("🟢 Health Check:")
for k, v in health.items():
    print(f"   {k:15}: {v}")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: /health is a Kubernetes readiness probe.")
    print("   K8s calls this endpoint every 10s. If it fails, traffic is rerouted.")

🟢 Health Check:
   status         : online
   redis          : connected
   model          : sentence-transformers/all-MiniLM-L6-v2
   version        : 2.0.0


In [4]:
# Cell 2: POST /ingest — load the knowledge base into the API
KNOWLEDGE_BASE = {
    "doc_aether_x1":  "The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU with a proprietary liquid-cooling system. It has a 4K OLED 120Hz ProMotion display, 64GB DDR5 RAM, and 2TB NVMe SSD. The price is $2,499 with a 3-year global warranty.",
    "doc_sony_xm5":   "The Sony WH-1000XM5 headphones feature industry-leading active noise cancellation with dual-chip processing. Battery life is 30 hours with ANC on, 40 hours without. Supports LDAC, AAC, and SBC codecs. Price is $349.",
    "doc_nebula_tab": "The Nebula Tab Pro is a 12-inch OLED tablet with an Apple M3 chip. Storage options: 256GB, 512GB, 1TB. Starting price $899.",
    "doc_warranty":   "All Aether products come with a 3-year global warranty covering manufacturing defects. AetherCare+ for $99/year covers accidental damage.",
    "doc_shipping":   "Standard shipping takes 3-5 business days. Express delivery (1-2 days) is $15. Free shipping on orders over $500.",
    "doc_returns":    "Products can be returned within 30 days in original condition. Refunds processed within 5-7 business days.",
}

print("📤 Ingesting knowledge base...")
for doc_id, text in KNOWLEDGE_BASE.items():
    resp = httpx.post(f"{API_URL}/ingest",
                      json={"parent_id": doc_id, "text": text},
                      timeout=30)
    result = resp.json()
    print(f"   {doc_id:20} → {result['propositions_indexed']} propositions indexed")

print("\n✅ Knowledge base loaded")

📤 Ingesting knowledge base...
   doc_aether_x1        → 3 propositions indexed
   doc_sony_xm5         → 4 propositions indexed
   doc_nebula_tab       → 3 propositions indexed
   doc_warranty         → 2 propositions indexed
   doc_shipping         → 3 propositions indexed
   doc_returns          → 2 propositions indexed

✅ Knowledge base loaded


In [5]:
# Cell 3: GET /stats — see the current state of the system
stats = httpx.get(f"{API_URL}/stats").json()

print("📊 System Stats:")
for k, v in stats.items():
    print(f"   {k:25}: {v}")

📊 System Stats:
   total_requests           : 0
   cache_hits               : 0
   cache_misses             : 0
   cache_hit_rate           : 0.0
   latency_p50_ms           : 0
   latency_p95_ms           : 0
   latency_p99_ms           : 0
   pdr_docs_indexed         : 37
   cache_threshold          : 0.25
   guard_threshold          : 0.35


## 🔍 Demo: The Cache MISS → HIT Cycle

In [6]:
# Cell 4: First chat — MISS path (PDR + LLM)
def chat(question: str, user_id: str = "student_01", session_id: str = None) -> dict:
    payload = {"user_id": user_id, "question": question}
    if session_id:
        payload["session_id"] = session_id
    resp = httpx.post(f"{API_URL}/chat", json=payload, timeout=30)
    return resp.json()

Q = "What GPU does the Aether X1 have?"

# First call
t0 = time.perf_counter()
result1 = chat(Q)
t1 = (time.perf_counter() - t0) * 1000

print("📨 First call (MISS — PDR + LLM):")
print(f"   Question   : {Q}")
print(f"   Cache hit  : {result1['cache_hit']}")
print(f"   Intent     : {result1['intent']}")
print(f"   Source     : {result1['source']}")
print(f"   Confidence : {result1['confidence']}")
print(f"   Latency    : {result1['latency_ms']}ms (server) / {t1:.1f}ms (total round-trip)")
print(f"   Answer     : {result1['answer'][:120]}...")

📨 First call (MISS — PDR + LLM):
   Question   : What GPU does the Aether X1 have?
   Cache hit  : False
   Intent     : product_specs
   Source     : doc_aether_x1
   Confidence : 0.783
   Latency    : 70.65ms (server) / 97.0ms (total round-trip)
   Answer     : Based on our product documentation:

The Aether X1 Flagship laptop features an NVIDIA RTX 4090 GPU with a proprietary li...


In [7]:
# Cell 5: Same question, rephrased — should HIT the cache
REPHRASINGS = [
    "What is the GPU in the Aether X1?",
    "Aether X1 GPU specs?",
    "Does the Aether X1 have a good GPU?",
]

print("📨 Rephrased questions (should be CACHE HIT):")
print(f"  {'Question':45} {'HIT':5} {'Latency':>10}")
print("-" * 65)
for q in REPHRASINGS:
    t0 = time.perf_counter()
    r = chat(q)
    latency = (time.perf_counter() - t0) * 1000
    hit_str = "💚 YES" if r['cache_hit'] else "❌  NO"
    print(f"  {q:45} {hit_str:5} {latency:>10.1f}ms")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: Cache HITs are <10ms server-side. Round-trip adds network overhead.")
    print("   In production with same datacenter, round-trip would be ~2-5ms.")

📨 Rephrased questions (should be CACHE HIT):
  Question                                      HIT      Latency
-----------------------------------------------------------------
  What is the GPU in the Aether X1?             💚 YES      130.3ms
  Aether X1 GPU specs?                          💚 YES       36.5ms
  Does the Aether X1 have a good GPU?           💚 YES       45.9ms


In [8]:
# Cell 6: Out-of-scope question — Hallucination Guard
out_of_scope = [
    "What is the best restaurant in Bangkok?",
    "Explain quantum computing in simple terms.",
    "What is the current Bitcoin price?"
]

print("🚨 Hallucination Guard test — out-of-scope questions:")
for q in out_of_scope:
    r = chat(q)
    blocked = "🔴 BLOCKED" if r['confidence'] is None or r['confidence'] == 0.0 else "🟢 ANSWERED"
    print(f"  {blocked} | {q}")
    print(f"           Answer: {r['answer'][:90]}\n")

🚨 Hallucination Guard test — out-of-scope questions:
  🔴 BLOCKED | What is the best restaurant in Bangkok?
           Answer: 🚨 I could not find reliable information to answer that question. Please contact support or

  🔴 BLOCKED | Explain quantum computing in simple terms.
           Answer: 🚨 I could not find reliable information to answer that question. Please contact support or

  🔴 BLOCKED | What is the current Bitcoin price?
           Answer: 🚨 I could not find reliable information to answer that question. Please contact support or



In [9]:
# Cell 7: Session-based memory — multi-turn conversation
SESSION = "demo_session_01"

TURNS = [
    "My Aether X1 has been overheating during gaming sessions.",
    "The fan runs at maximum speed even during normal web browsing.",
    "Is this overheating covered under the Aether warranty?",
]

print("💬 Multi-turn conversation with memory (session_id='demo_session_01'):")
for i, turn in enumerate(TURNS, 1):
    r = chat(turn, user_id="student_01", session_id=SESSION)
    print(f"\n  Turn {i}: {turn}")
    print(f"  Intent : {r['intent']}")
    print(f"  Answer : {r['answer'][:130]}...")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: Each turn is stored in SemanticMessageHistory in Redis.")
    print("   On future turns, relevant past turns are retrieved and injected into the prompt.")

💬 Multi-turn conversation with memory (session_id='demo_session_01'):

  Turn 1: My Aether X1 has been overheating during gaming sessions.
  Intent : technical_support
  Answer : 🚨 I could not find reliable information to answer that question. Please contact support or rephrase your query....

  Turn 2: The fan runs at maximum speed even during normal web browsing.
  Intent : product_specs
  Answer : 🚨 I could not find reliable information to answer that question. Please contact support or rephrase your query....

  Turn 3: Is this overheating covered under the Aether warranty?
  Intent : technical_support
  Answer : 🚨 I could not find reliable information to answer that question. Please contact support or rephrase your query....


In [10]:
# Cell 8: Cache warmup — fire N unique questions, then measure hit rate
WARMUP_QUESTIONS = [
    "How long does Sony XM5 battery last?",
    "What is the price of Sony XM5?",
    "Does Nebula Tab Pro support stylus?",
    "How long is the Aether warranty?",
    "What is the return policy?",
    "How much is express shipping?",
    "What storage does the Nebula Tab Pro have?",
    "Does Aether X1 have OLED display?",
]

# Phase 1: Warmup (all MISS)
print("Phase 1: Cache warmup...")
for q in WARMUP_QUESTIONS:
    chat(q)

# Phase 2: Repeat with slight rephrasings (should HIT)
REPEAT_QUESTIONS = [
    "Sony XM5 battery hours?",
    "Price of Sony WH-1000XM5?",
    "Nebula Tab Pro pen support?",
    "Aether warranty duration?",
    "Return period for purchases?",
    "Cost of fast shipping?",
]

print("Phase 2: Repeat queries (expect cache HITs)...")
hits = 0
for q in REPEAT_QUESTIONS:
    r = chat(q)
    if r['cache_hit']:
        hits += 1

hit_rate = hits / len(REPEAT_QUESTIONS) * 100
print(f"\n  Warmup cache hit rate: {hit_rate:.0f}% ({hits}/{len(REPEAT_QUESTIONS)})")

stats = httpx.get(f"{API_URL}/stats").json()
print(f"  Server-reported hit rate: {float(stats['cache_hit_rate'])*100:.1f}%")

Phase 1: Cache warmup...
Phase 2: Repeat queries (expect cache HITs)...

  Warmup cache hit rate: 17% (1/6)
  Server-reported hit rate: 16.7%


In [11]:
# Cell 9: Latency analysis — p50, p95, p99
LATENCY_QUESTIONS = [
    "What GPU does Aether X1 have?",
    "Sony XM5 battery?",
    "Nebula Tab Pro price?",
    "Aether warranty?",
    "Return policy?"
] * 4   # 20 total requests (questions already cached)

latencies = []
for q in LATENCY_QUESTIONS:
    t0 = time.perf_counter()
    chat(q)
    latencies.append((time.perf_counter() - t0) * 1000)

latencies.sort()
print("📊 Latency Distribution (20 requests, questions already cached):")
print(f"   p50 (median) : {statistics.median(latencies):>8.1f}ms")
print(f"   p95          : {latencies[int(len(latencies)*0.95)]:>8.1f}ms")
print(f"   p99          : {latencies[int(len(latencies)*0.99)]:>8.1f}ms")
print(f"   min          : {min(latencies):>8.1f}ms")
print(f"   max          : {max(latencies):>8.1f}ms")

📊 Latency Distribution (20 requests, questions already cached):
   p50 (median) :     34.0ms
   p95          :     87.9ms
   p99          :     87.9ms
   min          :     28.9ms
   max          :     87.9ms


In [12]:
# Cell 10: DELETE /cache — verify reset behaviour
resp = httpx.delete(f"{API_URL}/cache")
print("🗑️  Cache flushed:", resp.json())

# Re-run a cached question — should now MISS again
r = chat("What GPU does Aether X1 have?")
print(f"\nAfter flush — cache_hit: {r['cache_hit']} (should be False)")

🗑️  Cache flushed: {'status': 'cache_flushed', 'message': 'Semantic cache cleared.'}

After flush — cache_hit: False (should be False)


## ✅ Checkpoint

In [13]:
# Cell 11: Checkpoint
score = 0

# Test 1: health returns 'online'
h = httpx.get(f"{API_URL}/health").json()
if h.get('status') == 'online':
    print("✅ Test 1 PASS: API is online")
    score += 1
else:
    print("❌ Test 1 FAIL: API not online")

# Test 2: warmup hit rate
if hit_rate >= 50:
    print(f"✅ Test 2 PASS: Cache hit rate {hit_rate:.0f}% after warmup")
    score += 1
else:
    print(f"❌ Test 2 FAIL: Hit rate {hit_rate:.0f}% too low — check CACHE_THRESHOLD")

# Test 3: cache was successfully flushed
if not r['cache_hit']:
    print("✅ Test 3 PASS: DELETE /cache correctly reset the cache")
    score += 1
else:
    print("❌ Test 3 FAIL: Cache should have been empty after flush")

print(f"\n🏆 Score: {score}/3")

✅ Test 1 PASS: API is online
❌ Test 2 FAIL: Hit rate 17% too low — check CACHE_THRESHOLD
✅ Test 3 PASS: DELETE /cache correctly reset the cache

🏆 Score: 2/3


---
## 📝 Assignment: Load Test & Find the Breaking Point

**Task 1**: Implement `single_chat_request()` — async version that tracks full metadata.

**Task 2**: Implement `load_test()` — fire N requests at C concurrency and print a performance report.

**Task 3**: Implement `find_breaking_point()` — gradually increase concurrency until the API degrades.

**Task 4 (Bonus)**: Write 3 engineering recommendations based on your load test results.


In [4]:
# ── ASSIGNMENT ────────────────────────────────────────────────────────────────
import httpx
import asyncio
import time
import statistics

QUESTIONS_POOL = [
    "What GPU does Aether X1 have?",
    "Sony XM5 battery life?",
    "Nebula Tab Pro price?",
    "Aether warranty coverage?",
    "Return policy for products?",
    "Express shipping cost?",
    "Does Aether X1 have OLED display?",
    "What codecs does Sony XM5 support?",
]

async def single_chat_request(client: httpx.AsyncClient, question: str, 
                              user_id: str = "load_test_user") -> dict:
    """
    Fire a single POST /chat request asynchronously.
    """
    payload = {"user_id": user_id, "question": question}
    t0 = time.perf_counter()
    
    try:
        # Assuming API_URL is defined globally in the notebook (e.g., http://localhost:8000)
        ## ADD YOUR CODE HERE
        resp = None
        
        latency = (time.perf_counter() - t0) * 1000
        
        if resp.status_code == 200:
            data = resp.json()
            return {
                "question": question,
                "status_code": resp.status_code,
                "latency_ms": latency,
                "cache_hit": data.get("cache_hit", False),
                "answer_length": len(data.get("answer", ""))
            }
        else:
            return {
                "question": question,
                "status_code": resp.status_code,
                "latency_ms": latency,
                "cache_hit": False,
                "answer_length": 0,
                "error": f"HTTP {resp.status_code}"
            }
            
    except Exception as e:
        latency = (time.perf_counter() - t0) * 1000
        return {
            "question": question,
            "status_code": 500,
            "latency_ms": latency,
            "cache_hit": False,
            "answer_length": 0,
            "error": str(e)
        }

async def load_test(questions: list, concurrency: int = 10, 
                    total_requests: int = 100, silent: bool = False) -> dict:
    """
    Run a load test with a specified concurrency level.
    """
    # Round-robin the questions pool to hit the exact total_requests target
    queries = [questions[i % len(questions)] for i in range(total_requests)]
    results = []
    
    t_start = time.perf_counter()
    
    async with httpx.AsyncClient() as client:
        # Process in parallel batches of size `concurrency`
        ## ADD YOUR CODE HERE
        pass
            
    total_time = time.perf_counter() - t_start
    
    # Calculate performance metrics
    latencies = sorted([r["latency_ms"] for r in results])
    p50 = latencies[int(len(latencies) * 0.50)] if latencies else 0
    p95 = latencies[int(len(latencies) * 0.95)] if latencies else 0
    p99 = latencies[int(len(latencies) * 0.99)] if latencies else 0
    mean_latency = statistics.mean(latencies) if latencies else 0
    
    cache_hits = sum(1 for r in results if r.get("cache_hit"))
    errors = sum(1 for r in results if r.get("status_code") != 200 or "error" in r)
    
    rps = total_requests / total_time if total_time > 0 else 0
    cache_hit_rate = cache_hits / total_requests
    error_rate = errors / total_requests
    
    if not silent:
        print(f"\n--- Load Test Summary ({total_requests} requests @ concurrency {concurrency}) ---")
        print(f"Total Time      : {total_time:.2f}s")
        print(f"Throughput (RPS): {rps:.2f} req/s")
        print(f"Latencies       : p50={p50:.1f}ms, p95={p95:.1f}ms, p99={p99:.1f}ms")
        print(f"Cache Hit Rate  : {cache_hit_rate*100:.1f}%")
        print(f"Error Rate      : {error_rate*100:.1f}%")
    
    return {
        "total_time": total_time,
        "rps": rps,
        "mean_latency": mean_latency,
        "p50": p50,
        "p95": p95,
        "p99": p99,
        "cache_hit_rate": cache_hit_rate,
        "error_rate": error_rate
    }


async def find_breaking_point(questions: list, 
                              concurrency_levels: list = None) -> int:
    """
    Gradually increase concurrency to find the system's breaking point.
    """
    if concurrency_levels is None:
        concurrency_levels = [1, 5, 10, 20, 50]
        
    print(f"\n{'Concurrency':<12} | {'RPS':<6} | {'p95 Latency':<12} | {'Error Rate':<11} | {'Status'}")
    print("-" * 65)
    
    last_stable = 0
    
    for c in concurrency_levels:
        # Run test silently so we only print the table
        ## ADD YOUR CODE HERE
        pass
        
        # Determine if the system is breaking
        is_breaking = stats["error_rate"] > 0.05 or stats["mean_latency"] > 3000
        status_marker = "🔴 BREAKING" if is_breaking else "🟢 OK"
        
        print(f"{c:<12} | {stats['rps']:<6.1f} | {stats['p95']:<10.1f}ms | {stats['error_rate']*100:<10.1f}% | {status_marker}")
        
        if is_breaking:
            break
            
        last_stable = c
        
    return last_stable

# ── RUN TESTS ──────────────────────────────────────────────────────────────────
# NOTE: Ensure your FastAPI docker container is running before executing this cell.
print("🚀 Starting Load Test...")
await load_test(QUESTIONS_POOL, concurrency=5, total_requests=50)

print("\n📈 Finding Breaking Point...")
breaking_point = await find_breaking_point(QUESTIONS_POOL)
print(f"\n🏁 Final Breaking point: concurrency={breaking_point}")

🚀 Starting Load Test...

--- Load Test Summary (50 requests @ concurrency 5) ---
Total Time      : 1.33s
Throughput (RPS): 37.66 req/s
Latencies       : p50=118.7ms, p95=177.8ms, p99=179.1ms
Cache Hit Rate  : 0.0%
Error Rate      : 0.0%

📈 Finding Breaking Point...

Concurrency  | RPS    | p95 Latency  | Error Rate  | Status
-----------------------------------------------------------------
1            | 39.8   | 36.5      ms | 0.0       % | 🟢 OK
5            | 35.3   | 248.3     ms | 0.0       % | 🟢 OK
10           | 39.9   | 294.5     ms | 0.0       % | 🟢 OK
20           | 38.1   | 575.8     ms | 0.0       % | 🟢 OK
50           | 39.5   | 1238.8    ms | 0.0       % | 🟢 OK

🏁 Final Breaking point: concurrency=50


### 📝 Post-Load Test Analysis

1. **What was the maximum stable RPS your API handled?**
   *(Example Response)*: The API handled up to **~18 RPS** consistently before request failures and massive latency spikes began to occur at concurrency level 20. 

2. **At what concurrency did p95 latency exceed 1000ms?**
   *(Example Response)*: The p95 latency breached the 1-second mark around **concurrency = 20**. Up until concurrency 10, the p95 latency remained under 400ms.

3. **What is the cache hit rate under load? Why does it matter?**
   *(Example Response)*: The cache hit rate was roughly **87%** because we repeatedly fed the same 8 questions into the pool of 50 total requests. A high cache hit rate is crucial because semantic caching bypasses both the Vector DB retrieval and the LLM generation phase, reducing operation times from ~500ms down to ~15ms, saving heavy compute costs under load.

4. **Write 3 engineering changes that could improve throughput:**
   * **Implement horizontal scaling**: Add multiple FastAPI instances behind a load balancer (e.g., NGINX) to handle higher concurrent connections without bottle-necking a single thread.
   * **Expand the semantic cache TTL/threshold constraints**: Tuning the `distance_threshold` to encompass more phrasing variations could increase the hit rate, shifting the compute burden off the LLM and increasing RPS.
   * **Optimize the embedding latency**: Batching inputs directly within the API router rather than processing user strings individually (or swapping out the SentenceTransformer model to a faster quantized ONNX equivalent) could lower the cold-miss penalty.